In [56]:
import yfinance as yf
import numpy as np

In [57]:
tickers = ["AAPL", "MSFT", "TSLA", "GOOGL", "META", "NVDA", "AMZN", "PLTR", "RBLX"]
ohlcv_data = {}

for ticker in tickers:
    stocks = yf.download(ticker, period='1mo', interval='5m')
    stocks.dropna(how='any', inplace=True)
    stocks.columns = stocks.columns.droplevel(1)
    ohlcv_data[ticker] = stocks

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [58]:
def ATR(DF, n=14):
    df = DF.copy()
    df["H-L"] = df["High"] - df["Low"]
    df["H-PC"] = abs(df["High"] - df["Close"].shift(1))
    df["L-PC"] = abs(df["Low"] - df["Close"].shift(1))
    df["TR"] = df[["H-L", "H-PC","L-PC" ]].max(axis=1, skipna=False)
    df["ATR"] = df["TR"].ewm(com=n, min_periods=n).mean()
    return df["ATR"]

def breakout_rule(DF):
    df = DF.copy()
    df["resistance"] = df["High"].rolling(20).max().shift(1)
    df["support"] = df["Low"].rolling(20).min().shift(1)
    df["ATR"] = ATR(df, 20)
    df["Buy"]        = df["Close"] > df["resistance"]
    df["Sell"]       = df["Close"] < df["support"]
    return df

In [59]:
def strategy_return(DF):
    df = DF.copy()
    signal = ""
    returns = [0]

    for i in range(1, len(df)):
        if signal == "":                                    
            returns.append(0)
            if df["Buy"].iloc[i]:
                signal = "Buy"
            elif df["Sell"].iloc[i]:
                signal = "Sell"

        elif signal == "Buy":
            stop = df["Close"].iloc[i-1] - df["ATR"].iloc[i-1]        
            if df["Low"].iloc[i] < stop:
                signal = ""                                 
                returns.append((stop / df["Close"].iloc[i-1]) - 1)
            else:
                returns.append((df["Close"].iloc[i] / df["Close"].iloc[i-1]) - 1)

        elif signal == "Sell":                              
            stop = df["Close"].iloc[i-1] + df["ATR"].iloc[i-1]                              
            if df["High"].iloc[i] > stop:
                signal = "" 
                returns.append((df["Close"].iloc[i-1] / stop) - 1)
            else:
                returns.append((df["Close"].iloc[i-1] / df["Close"].iloc[i]) - 1)

    df["ret"] = returns
    return df

In [60]:
BARS_PER_YEAR = 252 * 78   
def CAGR(DF):
    df = DF.copy()
    df["cum_return"] = (1 + df["ret"]).cumprod()
    n = len(df) / BARS_PER_YEAR
    return df["cum_return"].iloc[-1] ** (1/n) - 1

def volatility(DF):
    df = DF.copy()
    return df["ret"].std() * np.sqrt(BARS_PER_YEAR)

def sharpe(DF, rf=0.03):
    return (CAGR(DF) - rf) / volatility(DF)

def max_dd(DF):
    df = DF.copy()
    df["cum_return"]   = (1 + df["ret"]).cumprod()
    df["cum_roll_max"] = df["cum_return"].cummax()
    df["drawdown"]     = (df["cum_roll_max"] - df["cum_return"]) / df["cum_roll_max"]
    return df["drawdown"].max()

strategy_df = {}

for ticker in ohlcv_data:
    strategy_df[ticker] = strategy_return(breakout_rule(ohlcv_data[ticker]))

print("Strategy Returns (breakout + ATR):\n")
for ticker in strategy_df:
    df = strategy_df[ticker]
    ret_total = (1 + df["ret"]).prod() - 1
    print(f"{ticker}")
    print(f"  Total Return  : {ret_total*100:.2f}%")
    print(f"  CAGR           : {CAGR(df)*100:.2f}%")
    print(f"  Sharpe         : {sharpe(df):.2f}")
    print(f"  Max Drawdown   : {max_dd(df)*100:.2f}%\n")

Strategy Returns (breakout + ATR):

AAPL
  Total Return  : 4.32%
  CAGR           : 70.41%
  Sharpe         : 3.63
  Max Drawdown   : 4.61%

MSFT
  Total Return  : 2.54%
  CAGR           : 37.25%
  Sharpe         : 1.92
  Max Drawdown   : 2.90%

TSLA
  Total Return  : 10.83%
  CAGR           : 265.22%
  Sharpe         : 11.24
  Max Drawdown   : 7.61%

GOOGL
  Total Return  : 4.90%
  CAGR           : 82.76%
  Sharpe         : 4.36
  Max Drawdown   : 6.03%

META
  Total Return  : 13.77%
  CAGR           : 408.18%
  Sharpe         : 10.51
  Max Drawdown   : 6.04%

NVDA
  Total Return  : 0.93%
  CAGR           : 12.31%
  Sharpe         : 0.48
  Max Drawdown   : 6.33%

AMZN
  Total Return  : 5.73%
  CAGR           : 101.89%
  Sharpe         : 4.76
  Max Drawdown   : 3.92%

PLTR
  Total Return  : 7.59%
  CAGR           : 151.51%
  Sharpe         : 4.91
  Max Drawdown   : 4.32%

RBLX
  Total Return  : 16.33%
  CAGR           : 572.68%
  Sharpe         : 11.65
  Max Drawdown   : 7.37%

